# DriftPA — Training Notebook

**OpenEnv Hackathon | Track 3.2 + Patronus AI**

Trains a personal assistant LLM agent on the DriftPA environment using **Unsloth** + **TRL GRPOTrainer**.

### Four Novel Mechanics
1. **Schema Drift** — API field names change mid-episode; agent must call `list_tools()` to adapt
2. **Time Pressure** — tasks expire if not handled within N steps (cascade failures)
3. **Irreversible Actions** — `book`, `reply`, `cancel` cannot be undone; wrong actions cause cascades
4. **Cancellation Window** — policy controls how late a booking can be cancelled (tightens post-drift)

### Training Approach
We use **TRL GRPOTrainer** (Group Relative Policy Optimization) at the step level:
- Each environment observation (at a diverse episode stage) is a GRPO prompt
- GRPO samples `num_generations=8` candidate actions per observation
- The DriftPA environment **replays to the exact mid-episode state** and evaluates each candidate
- GRPO updates the model to prefer higher-reward actions via group-relative normalization

> **Dataset diversity is critical**: we sample states at steps 0–6, including post-drift states
> where schema adaptation (+2) and policy checking (+1) rewards fire. Training only on step-0 states
> teaches the model that stale schema fields work (they do pre-drift, but are catastrophic post-drift).

## 1. Install Dependencies

In [ ]:
%%capture
# Unsloth: fast model loading + LoRA (H100 bf16; set load_in_4bit=True for T4 fallback)
!pip install unsloth
# TRL: GRPOTrainer for policy optimisation
!pip install "trl>=0.11.0" datasets
# DriftPA environment
!pip install "openenv-core>=0.2.1" matplotlib numpy

In [ ]:
import os, sys

REPO_URL = "https://github.com/rajashekarcs2023/openenv-RL"
REPO_DIR = "/content/openenv-RL"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready at", REPO_DIR)

## 2. Load Model (Qwen2.5-14B on H100 — full bf16, no quantization needed)

In [ ]:
import torch
from unsloth import FastLanguageModel

# H100 (80GB): run Qwen2.5-14B in full bf16 — no quantization needed.
# 14B in bf16 = ~28GB weights; with LoRA + GRPO buffer comfortably fits in 80GB.
# (Change to "unsloth/Qwen2.5-7B-Instruct" + load_in_4bit=True for T4 fallback)
MODEL_NAME  = "unsloth/Qwen2.5-14B-Instruct"
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,       # bf16 on H100 — much better training quality
    dtype=torch.bfloat16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,                     # larger rank → more capacity on H100
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME} (bf16, H100)")
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 3. Environment + Utilities

In [ ]:
import re
import torch
from driftpa.server.environment import DriftPAEnvironment
from driftpa.models import DriftPAAction

# Verify environment works
env = DriftPAEnvironment()
obs = env.reset(seed=0)
print(f"Environment OK — inbox: {len(obs.inbox)} msgs, calendar: {len(obs.calendar)} events")
print(f"Schema version: {env.state.schema_version}, Step: {obs.time_step}")

In [ ]:
def format_prompt(obs) -> str:
    """Encode environment observation as a text prompt for the LLM."""
    inbox_lines = "\n".join(
        f"  [{m['id']}] {m['sender']}: {m['subject']} "
        f"[priority={m['priority']} expires_step={m.get('expires_at_step','none')}]"
        for m in obs.inbox
    )
    cal_lines = "\n".join(
        f"  [{e['id']}] {e.get('title') or e.get('event_name','?')} "
        f"@ {e.get('begins_at') or e.get('start_time','?')}"
        for e in obs.calendar
    )
    tool_lines = "\n".join(
        f"  {name}({', '.join(f'{p}:{t}' for p,t in s.get('params',{}).items())})"
        + (" [IRREVERSIBLE]" if name in {"reply_message","book_restaurant","book_ride","cancel_booking","confirm_booking"} else "")
        for name, s in obs.available_tools.items()
    )
    policy_str = ", ".join(f"{k}={v}" for k, v in obs.policy.items())
    urgent_str = f"⚠ URGENT expiring soon: {obs.urgent_expiring}" if obs.urgent_expiring else "No urgent tasks"

    return (
        f"You are a personal executive assistant. Step {obs.time_step}/15.\n"
        f"INBOX:\n{inbox_lines}\n"
        f"CALENDAR:\n{cal_lines}\n"
        f"TOOLS (field names may drift — always check):\n{tool_lines}\n"
        f"POLICY: {policy_str}\n"
        f"{urgent_str}\n"
        f"LAST: {obs.last_action_result[:120]}\n"
        f"Action (format: TOOL_NAME(key=value, ...)):"
    )


def parse_action(text: str) -> DriftPAAction:
    """Parse LLM output → DriftPAAction. Defaults to list_tools() on parse error."""
    # Scan lines from bottom for the first tool_name(...) pattern
    for line in reversed(text.strip().split("\n")):
        line = line.strip().rstrip("!")
        m = re.match(r'^(\w+)\((.*)\)\s*$', line)
        if not m:
            continue
        tool_name = m.group(1)
        payload   = {}
        for pair in m.group(2).split(","):
            if "=" in pair:
                k, v = pair.split("=", 1)
                v = v.strip().strip('"\'')
                try:
                    payload[k.strip()] = int(v)
                except ValueError:
                    payload[k.strip()] = v
        return DriftPAAction(tool_name=tool_name, payload=payload)
    return DriftPAAction(tool_name="list_tools", payload={})

In [ ]:
def run_episode(env: DriftPAEnvironment, model, tokenizer,
                greedy: bool = False, seed: int = None) -> tuple[list, float]:
    """Run one full episode. Returns (trajectory, total_reward)."""
    obs = env.reset(seed=seed)
    trajectory   = []
    total_reward = 0.0

    while not obs.done and env.state.step_count < 15:
        prompt = format_prompt(obs)
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=MAX_SEQ_LEN,
        ).to(model.device)

        # Build generation kwargs — only pass temperature when sampling
        # (passing temperature=None with do_sample=False causes warnings in some HF versions)
        gen_kwargs = dict(
            max_new_tokens=128,
            do_sample=not greedy,
            pad_token_id=tokenizer.eos_token_id,
        )
        if not greedy:
            gen_kwargs["temperature"] = 0.8

        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)

        new_toks = out[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(new_toks, skip_special_tokens=True).strip()

        action      = parse_action(response)
        obs         = env.step(action)
        step_reward = float(obs.reward or 0.0)
        total_reward += step_reward

        trajectory.append({"prompt": prompt, "response": response, "reward": step_reward})

        if action.tool_name == "finish":
            break

    # Force episode close
    if not obs.done:
        obs = env.step(DriftPAAction(tool_name="finish", payload={}))
        total_reward += float(obs.reward or 0.0)

    return trajectory, total_reward

## 4. Baseline Evaluation (untrained model)

In [ ]:
BASELINE_EPS = 20
model.eval()
baseline_rewards = []

print(f"Running {BASELINE_EPS} baseline episodes (untrained)...\n")
for i in range(BASELINE_EPS):
    _, r = run_episode(env, model, tokenizer)
    baseline_rewards.append(r)
    print(f"  ep {i+1:02d}: {r:+.1f}")

baseline_mean = sum(baseline_rewards) / len(baseline_rewards)
print(f"\nBaseline mean: {baseline_mean:.2f}  (expect strongly negative)")

## 5. Training with TRL GRPOTrainer

GRPO (Group Relative Policy Optimization) works by:
1. Sampling `num_generations` candidate actions for the same observation
2. Getting environment rewards for each candidate
3. Normalising rewards within the group (relative comparison)
4. Updating the policy to prefer higher-reward actions

Applied here at **step level**: each environment observation is one GRPO prompt.

In [ ]:
import json
import random as _random
from datasets import Dataset

def build_grpo_dataset(n_prompts: int = 1000, max_advance_steps: int = 6) -> Dataset:
    """
    Build a dataset of DIVERSE mid-episode environment states for GRPO training.

    WHY DIVERSITY MATTERS:
      At step 0 (pre-drift), stale restaurant schema gives +2 reward (it works pre-drift).
      At step 5 (post-drift), the same call gives -6 (policy violation + stale schema).
      Training only on step-0 states teaches the model exactly the wrong behavior.
      We must sample states at steps 0–6, spanning both sides of the drift trigger.

    HOW STATE REPLAY WORKS:
      We record the exact sequence of safe (non-irreversible) actions taken to reach
      each state. The reward function replays those actions in a fresh environment to
      reconstruct the exact mid-episode context before evaluating the candidate action.
      This ensures the reward signal is computed in the correct state, not step 0.

    Dataset columns:
      prompt        — the formatted observation at the sampled state
      seed          — episode seed (deterministic scenario)
      actions_taken — JSON list of {tool, payload} actions to replay
    """
    env_ds = DriftPAEnvironment()
    rows   = []
    n_seeds = 50  # scenario variety

    # Only safe (non-irreversible) actions for replay — deterministic, no side effects
    SAFE_TOOL_POOL = ["read_calendar", "list_tools", "query_policy"]

    for i in range(n_prompts):
        seed = i % n_seeds
        obs  = env_ds.reset(seed=seed)

        # Advance 0–max_advance_steps into the episode using safe actions.
        # This ensures ~equal representation of: pre-drift, drift-firing, post-drift states.
        n_advance      = _random.randint(0, max_advance_steps)
        actions_taken  = []

        for _ in range(n_advance):
            if obs.done:
                break

            # Prefer reading unread messages (information-rich, 0 reward, non-redundant)
            unread = [m["id"] for m in obs.inbox
                      if m["id"] not in env_ds._read_messages]
            if unread and _random.random() > 0.4:
                tool    = "read_message"
                payload = {"message_id": unread[0]}
            else:
                tool    = _random.choice(SAFE_TOOL_POOL)
                payload = {}

            actions_taken.append({"tool": tool, "payload": payload})
            obs = env_ds.step(DriftPAAction(tool_name=tool, payload=payload))

        rows.append({
            "prompt":        format_prompt(obs),
            "seed":          seed,
            "actions_taken": json.dumps(actions_taken),  # JSON string for HF Dataset
        })

    return Dataset.from_list(rows)


grpo_dataset = build_grpo_dataset(n_prompts=1000, max_advance_steps=6)
print(f"GRPO dataset: {len(grpo_dataset)} prompts across {min(50, len(grpo_dataset))} scenarios")

# Show distribution of episode stages sampled
stages = [len(json.loads(row["actions_taken"])) for row in grpo_dataset]
for s in range(7):
    count = stages.count(s)
    print(f"  Step {s} states: {count} ({100*count/len(stages):.0f}%)")

print("\nSample post-drift prompt (step ≥ 3, first 400 chars):")
post_drift = next(r for r in grpo_dataset if len(json.loads(r["actions_taken"])) >= 3)
print(post_drift["prompt"][:400])

In [ ]:
def driftpa_reward(completions, prompts, seed, actions_taken, **kwargs) -> list[float]:
    """
    TRL GRPOTrainer reward function with exact mid-episode state replay.

    CORRECTNESS: The reward function receives the same `seed` and `actions_taken`
    used to build each dataset row. It replays those actions in a fresh environment
    to reconstruct the exact state the prompt was generated from, then evaluates
    the candidate action. Without replay, we'd evaluate against step-0 state,
    giving wrong rewards for post-drift observations.

    Args:
        completions  : list[str]  — candidate action strings from the model
        prompts      : list[str]  — observation prompts (for debugging)
        seed         : list[int]  — episode seeds from dataset
        actions_taken: list[str]  — JSON-encoded action sequences to replay

    Returns: list[float] — step-level reward for each candidate completion
    """
    rewards  = []
    eval_env = DriftPAEnvironment()

    for completion, ep_seed, acts_json in zip(completions, seed, actions_taken):
        try:
            obs = eval_env.reset(seed=int(ep_seed))

            # Replay recorded safe actions to reach the correct mid-episode state
            acts = json.loads(acts_json) if isinstance(acts_json, str) else acts_json
            for act in acts:
                if not obs.done:
                    obs = eval_env.step(
                        DriftPAAction(tool_name=act["tool"], payload=act["payload"])
                    )

            # Evaluate the candidate action in the reconstructed state
            action = parse_action(completion)
            obs    = eval_env.step(action)
            r      = float(obs.reward or 0.0)

        except Exception:
            r = -1.0   # parse failure or env error → small penalty
        rewards.append(r)

    return rewards


# Sanity check: verify state replay works correctly
print("Sanity check — state replay reward function:")
# At step 0 (seed=0): reply to boss = +4 (urgent, before expiry)
r0 = driftpa_reward(
    completions=["reply_message(message_id=msg_boss, text=On it.)"],
    prompts=["..."], seed=[0], actions_taken=["[]"],
)
print(f"  Step-0, reply boss: {r0[0]:+.1f}  (expect +4.0)")

# At step 5 post-drift (seed=0): book_ride with correct schema = +1 (expiry -3 + correct +4)
acts_5 = json.dumps([
    {"tool": "read_message", "payload": {"message_id": "msg_boss"}},
    {"tool": "read_message", "payload": {"message_id": "msg_friend"}},
    {"tool": "read_message", "payload": {"message_id": "msg_finance"}},
    {"tool": "read_message", "payload": {"message_id": "msg_partner"}},
])
r5_correct = driftpa_reward(
    completions=["book_ride(eta_minutes=15, drop_off=Home)"],
    prompts=["..."], seed=[0], actions_taken=[acts_5],
)
r5_stale = driftpa_reward(
    completions=["book_ride(pickup_time=21:00, destination=Home)"],
    prompts=["..."], seed=[0], actions_taken=[acts_5],
)
print(f"  Step-5, book_ride correct schema: {r5_correct[0]:+.1f}  (expect +1.0)")
print(f"  Step-5, book_ride STALE schema:   {r5_stale[0]:+.1f}  (expect -3.0)")
print("Reward function verified — state replay working correctly.")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir          = "driftpa-grpo-out",
    num_train_epochs    = 3,      # H100: 3 full passes over 1000-prompt dataset
    # H100: 2 prompts × 8 candidates = 16 rollouts per update step
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,   # effective batch = 8 prompts × 8 = 64 rollouts
    num_generations     = 8,      # GRPO group size (vs 4 on T4) — better variance reduction
    max_prompt_length   = MAX_SEQ_LEN,
    max_completion_length = 128,  # more room for LLM to reason before action
    learning_rate       = 3e-6,   # slightly lower for 14B stability
    warmup_ratio        = 0.05,
    logging_steps       = 10,
    save_steps          = 100,
    report_to           = "none",
    # KL penalty: keeps policy close to reference (prevents reward hacking)
    kl_coef             = 0.04,   # slightly lower for 14B (already well-calibrated)
)

trainer = GRPOTrainer(
    model         = model,
    reward_funcs  = driftpa_reward,
    args          = grpo_config,
    train_dataset = grpo_dataset,
)

total_rollouts = len(grpo_dataset) * grpo_config.num_train_epochs * grpo_config.num_generations
print("GRPOTrainer ready (H100 configuration)")
print(f"  Model:         {MODEL_NAME}")
print(f"  Dataset:       {len(grpo_dataset)} diverse mid-episode states")
print(f"  Epochs:        {grpo_config.num_train_epochs}")
print(f"  Generations:   {grpo_config.num_generations} per prompt")
print(f"  Total rollouts:{total_rollouts:,}  (candidate actions evaluated against env)")
print(f"  LR:            {grpo_config.learning_rate}  |  KL coef: {grpo_config.kl_coef}")
print()

trainer.train()
print("\nTraining complete!")

## 6. Post-Training Evaluation

In [ ]:
POST_EPS = 20
model.eval()
post_rewards = []

print(f"Running {POST_EPS} post-training episodes...\n")
for i in range(POST_EPS):
    _, r = run_episode(env, model, tokenizer)
    post_rewards.append(r)
    print(f"  ep {i+1:02d}: {r:+.1f}")

post_mean = sum(post_rewards) / len(post_rewards)
print(f"\nPost-training mean:  {post_mean:.2f}")
print(f"Baseline mean:       {baseline_mean:.2f}")
print(f"Improvement:         {post_mean - baseline_mean:+.2f}")

## 7. Reward Curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract per-step training rewards from trainer logs.
# TRL logs rewards under different keys depending on version — try all common ones.
REWARD_KEYS = ["reward", "rewards", "mean_reward", "train/reward", "train/rewards"]
train_log_rewards = []
for entry in trainer.state.log_history:
    for key in REWARD_KEYS:
        if key in entry:
            train_log_rewards.append(float(entry[key]))
            break

if not train_log_rewards:
    # Fallback: look for any numeric entry that isn't a loss or epoch counter
    for entry in trainer.state.log_history:
        vals = [v for k, v in entry.items()
                if isinstance(v, (int, float)) and "loss" not in k and "epoch" not in k and "step" not in k]
        if vals:
            train_log_rewards.append(float(vals[0]))

print(f"Training log entries: {len(trainer.state.log_history)}")
print(f"Reward entries found: {len(train_log_rewards)}")
if train_log_rewards:
    print(f"Reward range: {min(train_log_rewards):.2f} to {max(train_log_rewards):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("DriftPA — Training Results (GRPO on H100)", fontsize=14, fontweight="bold")

# Left: GRPO step-level reward curve
if train_log_rewards:
    axes[0].plot(train_log_rewards, alpha=0.35, color="steelblue", label="Step reward")
    if len(train_log_rewards) >= 10:
        w = max(10, len(train_log_rewards) // 30)  # adaptive smoothing window
        smoothed = np.convolve(train_log_rewards, np.ones(w)/w, mode="valid")
        axes[0].plot(range(w-1, len(train_log_rewards)), smoothed,
                     color="steelblue", linewidth=2.5, label=f"Smoothed (w={w})")
else:
    axes[0].text(0.5, 0.5, "No reward data in trainer logs\n(check trainer.state.log_history keys)",
                 ha="center", va="center", transform=axes[0].transAxes, color="red")
axes[0].axhline(y=0, color="gray", linestyle=":", linewidth=0.8)
axes[0].set_xlabel("Training step")
axes[0].set_ylabel("Step reward")
axes[0].set_title("GRPO Training — Step-Level Rewards")
axes[0].legend()

# Right: episode-level before vs after bar chart
b_mean = np.mean(baseline_rewards)
p_mean = np.mean(post_rewards)
bars = axes[1].bar(
    ["Baseline\n(untrained)", "Post-GRPO\n(trained)"],
    [b_mean, p_mean],
    color=["#e74c3c", "#2ecc71"], alpha=0.85, width=0.5,
)
for bar, val in zip(bars, [b_mean, p_mean]):
    offset = 0.3 if val >= 0 else -1.8
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        val + offset,
        f"{val:.1f}", ha="center", va="bottom", fontweight="bold", fontsize=12,
    )
axes[1].axhline(y=0, color="gray", linestyle=":", linewidth=0.8)
axes[1].set_ylabel("Average Episode Reward")
axes[1].set_title("Episode Reward: Before vs After Training")

plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nBaseline: {b_mean:.2f}  |  Post-GRPO: {p_mean:.2f}  |  Δ = {p_mean-b_mean:+.2f}")
print("Saved: reward_curve.png")

## 8. Demo: Hero Scenario (seed=0)

Run the trained agent on the canonical scenario to show qualitative improvement.

In [ ]:
print("=" * 60)
print("HERO SCENARIO — Trained Agent (greedy, seed=0)")
print("=" * 60)

demo_env = DriftPAEnvironment()
model.eval()
traj, total_r = run_episode(demo_env, model, tokenizer, greedy=True, seed=0)

for i, step in enumerate(traj, 1):
    action = parse_action(step["response"])
    print(f"  Step {i:02d} | {action.tool_name:20s} | reward={step['reward']:+.1f}")

s = demo_env.state
print(f"\nTotal reward:     {total_r:+.1f}  (optimal = +22)")
print(f"Tasks resolved:   {s.tasks_resolved} / 3")
print(f"Tasks expired:    {s.tasks_expired}")
print(f"Cascade failures: {s.cascade_failures}")
print(f"Schema version:   {s.schema_version}  (drift events experienced)")
print()
print("Optimal sequence for reference:")
print("  1. read_message(msg_boss)")
print("  2. reply_message(msg_boss)   ← resolve urgent before step 4 expiry")
print("  3. reply_message(msg_friend) ← resolve before step 5 expiry")
print("  4. list_tools()              ← detect schema drift")
print("  5. query_policy()            ← detect policy drift (+1)")
print("  6. move_event(cal_teamsync, 17:30)")
print("  7. move_event(cal_investor, 18:00) ← resolve calendar conflict")
print("  8. book_ride(eta_minutes=15, ...) ← correct post-drift schema")
print("  9. reply_message(msg_partner)")
print(" 10. finish()                  → +22 total reward")